In [7]:
import pandas as pd
import numpy as np
import warnings

# Professional cleanup (matches teammate style)
warnings.filterwarnings("ignore")

# Load the raw datasets
orders_df = pd.read_csv('orders.csv')
items_df = pd.read_csv('order_items.csv')

# Merge them together (Matching teammate's standard merge logic)
# This allows us to know which items were sold in which cafeteria
df = pd.merge(items_df, orders_df, on='order_id')

# Check if it works (Matching teammate's preview style)
print("Merged Data Preview (Items and their Canteens):")
df[['order_id', 'cafeteria_id', 'item_id', 'quantity']].head(10)

Merged Data Preview (Items and their Canteens):


,order_id,cafeteria_id,item_id,quantity
0,1,1,7,1
1,1,1,16,1
2,1,1,17,1
3,2,3,49,1
4,2,3,52,1
5,2,3,53,1
6,3,2,19,1
7,4,2,26,1
8,4,2,34,1
9,4,2,35,1


In [8]:
import json
from mlxtend.frequent_patterns import apriori, association_rules

# 1. Generate Combo Rules (Apriori Analysis)

combo_rules = {}

# Use unique cafeteria IDs to process each canteen separately
for canteen_id in df['cafeteria_id'].unique():
    canteen_data = df[df['cafeteria_id'] == canteen_id]
    
    # Create the basket matrix (Professional Standard)
    basket = (canteen_data.groupby(['order_id', 'item_id'])['quantity']
              .sum().unstack().reset_index().fillna(0)
              .set_index('order_id'))
    
    # Python 3.14 Fix: Use .map() instead of the older .applymap()
    basket = basket.map(lambda x: 1 if x > 0 else 0)
    
    # Run Apriori to find frequent pairs
    frequent_itemsets = apriori(basket, min_support=0.03, use_colnames=True)
    
    if not frequent_itemsets.empty:
        # Generate association rules
        rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.2)
        
        # Filter for simple 1-to-1 item pairs (Professional filtering)
        rules = rules[(rules['antecedents'].apply(len) == 1) & (rules['consequents'].apply(len) == 1)]
        top_rules = rules.sort_values('confidence', ascending=False).head(10)
        
        # Save as a clean dictionary for JSON export
        combo_rules[int(canteen_id)] = [{"item_a": int(list(r['antecedents'])[0]), 
                                         "item_b": int(list(r['consequents'])[0])} 
                                        for _, r in top_rules.iterrows()]

# Save the combo rules 
with open('combo_rules.json', 'w') as f:
    json.dump(combo_rules, f, indent=4)
print("combo_rules.json saved successfully!")


# 2. Generate Failing Items & BOGO Targets

failing_items = {}
bogo_rules = {}

for canteen_id in df['cafeteria_id'].unique():
    canteen_data = df[df['cafeteria_id'] == canteen_id]
    item_sales = canteen_data.groupby('item_id')['quantity'].sum().reset_index()
    
    # Calculate thresholds
    low_sales_threshold = item_sales['quantity'].quantile(0.25)
    critical_sales_threshold = item_sales['quantity'].quantile(0.10)
    
    failing_items[str(canteen_id)] = {
        "items": [int(i) for i in item_sales[item_sales['quantity'] <= low_sales_threshold]['item_id']],
        "count": len(item_sales[item_sales['quantity'] <= low_sales_threshold])
    }
    
    bogo_rules[str(canteen_id)] = {
        "items": [int(i) for i in item_sales[item_sales['quantity'] <= critical_sales_threshold]['item_id']],
        "count": len(item_sales[item_sales['quantity'] <= critical_sales_threshold])
    }
    
# Save the final JSON files
with open('failing_items.json', 'w') as f:
    json.dump(failing_items, f, indent=4)
print("failing_items.json saved successfully!")

with open('bogo_rules.json', 'w') as f:
    json.dump(bogo_rules, f, indent=4)
print("bogo_rules.json saved successfully!\n")

print("--- Discount Strategy Generation Complete ---")

combo_rules.json saved successfully!
failing_items.json saved successfully!
bogo_rules.json saved successfully!

--- Discount Strategy Generation Complete ---
